# Projet AirBnB — Prediction de Prix par IA

**Objectif** : Mettre en forme exploitable pour une IA les donnees AirBnB afin de predire le prix optimal d'un nouveau logement.

| Ville | Date | Source |
|-------|------|--------|
| Lyon | 15 juin 2025 | insideairbnb.com |
| Paris | 06 juin 2025 | insideairbnb.com |
| Bordeaux | 15 juin 2025 | insideairbnb.com |

**Pipeline** : Collecte -> Pretraitement -> Analyse descriptive -> Recodage -> Aberrants -> Modelisation -> Prediction


## 1. Introduction & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os, re, urllib.request, gzip, shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CITIES = {'Lyon': 'lyon', 'Paris': 'paris', 'Bordeaux': 'bordeaux'}
COLORS = {'Lyon': 'steelblue', 'Paris': 'coral', 'Bordeaux': 'mediumseagreen'}
TARGET = 'price'

for key in CITIES.values():
    os.makedirs(f'data/raw/{key}', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('reports/figures', exist_ok=True)
print('Imports OK')

## 2. Collecte des Donnees

Telechargement des fichiers detailles (`listings.csv.gz`, 79 colonnes) depuis [insideairbnb.com](https://insideairbnb.com/get-the-data/).


In [ ]:
URLS = {
    'lyon':     'https://data.insideairbnb.com/france/auvergne-rhone-alpes/lyon/2025-06-15/data/listings.csv.gz',
    'paris':    'https://data.insideairbnb.com/france/ile-de-france/paris/2025-06-06/data/listings.csv.gz',
    'bordeaux': 'https://data.insideairbnb.com/france/nouvelle-aquitaine/bordeaux/2025-06-15/data/listings.csv.gz',
}
HEADERS = {'User-Agent': 'Mozilla/5.0'}

for city, url in URLS.items():
    dest = f'data/raw/{city}/listings_detail.csv'
    if os.path.exists(dest):
        print(f'[SKIP] {city} deja present')
        continue
    gz = dest + '.gz'
    print(f'[DL]   {city}...')
    req = urllib.request.Request(url, headers=HEADERS)
    with urllib.request.urlopen(req, timeout=120) as r, open(gz, 'wb') as f:
        f.write(r.read())
    with gzip.open(gz, 'rb') as g, open(dest, 'wb') as out:
        shutil.copyfileobj(g, out)
    os.remove(gz)
    print(f'       -> {os.path.getsize(dest):,} octets')
print('Collecte terminee.')

In [ ]:
raws = {}
for name, key in CITIES.items():
    raws[name] = pd.read_csv(f'data/raw/{key}/listings_detail.csv', low_memory=False)
    print(f'{name:10s} : {raws[name].shape[0]:,} annonces x {raws[name].shape[1]} colonnes')

## 3. Pretraitement des Donnees

### 3a. Selection des colonnes

Sur les 79 colonnes disponibles, on conserve **26 colonnes pertinentes** pour la prediction du prix. Les colonnes redondantes (URL, texte libre, scrape_id...) sont exclues.


In [ ]:
SELECTED = [
    'id', 'neighbourhood_cleansed', 'room_type', 'property_type',
    'accommodates', 'bedrooms', 'beds', 'bathrooms_text',
    'price', 'minimum_nights', 'maximum_nights', 'availability_365',
    'number_of_reviews', 'number_of_reviews_ltm', 'reviews_per_month',
    'review_scores_rating', 'review_scores_cleanliness',
    'review_scores_location', 'review_scores_value',
    'host_is_superhost', 'host_response_rate', 'host_acceptance_rate',
    'calculated_host_listings_count', 'instant_bookable',
    'latitude', 'longitude',
]

dfs = {}
for name, df in raws.items():
    cols = [c for c in SELECTED if c in df.columns]
    dfs[name] = df[cols].copy()
    print(f'{name:10s} : {dfs[name].shape[1]} colonnes selectionnees')

### 3b. Suppression des doublons

Suppression des lignes ayant le meme `id`.

In [ ]:
for name, df in dfs.items():
    n_dup = df.duplicated(subset=['id']).sum()
    print(f'{name:10s} : {n_dup} doublon(s)')
    if n_dup > 0:
        dfs[name] = df.drop_duplicates(subset=['id']).reset_index(drop=True)

### 3c. Gestion des valeurs manquantes

Remplissage par 0 pour les metriques absentes. Suppression des lignes sans prix.

In [ ]:
print('Valeurs manquantes avant traitement :')
for name, df in dfs.items():
    missing = df.isnull().sum()
    print(f'\n{name} :')
    print(missing[missing > 0].to_string())

In [ ]:
FILL = {
    'reviews_per_month': 0.0, 'number_of_reviews': 0,
    'number_of_reviews_ltm': 0, 'review_scores_rating': 0.0,
    'review_scores_cleanliness': 0.0, 'review_scores_location': 0.0,
    'review_scores_value': 0.0, 'bedrooms': 1.0, 'beds': 1.0,
}

for name, df in dfs.items():
    for col, val in FILL.items():
        if col in df.columns:
            df[col] = df[col].fillna(val)
    dfs[name] = df.dropna(subset=['price']).reset_index(drop=True)

print('Apres traitement :')
for name, df in dfs.items():
    print(f'  {name:10s} : {df.shape}')

### 3d. Conversion des types de colonnes

- `price` : texte `"$150.00"` -> float
- `bathrooms_text` : texte `"1 bath"` -> float
- `host_is_superhost` / `instant_bookable` : `"t"`/`"f"` -> 1/0
- Taux en % -> float numerique


In [ ]:
def extract_bathrooms(text):
    if pd.isna(text): return 1.0
    nums = re.findall(r'[\d.]+', str(text))
    return float(nums[0]) if nums else 1.0

for name, df in dfs.items():
    df['price'] = (df['price'].astype(str)
                   .str.replace(r'[$,]', '', regex=True)
                   .pipe(pd.to_numeric, errors='coerce'))
    if 'bathrooms_text' in df.columns:
        df['bathrooms'] = df['bathrooms_text'].apply(extract_bathrooms)
        df.drop(columns=['bathrooms_text'], inplace=True)
    for col in ['host_response_rate', 'host_acceptance_rate']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace('%','',regex=False),
                                    errors='coerce').fillna(0)
    for col in ['host_is_superhost', 'instant_bookable']:
        if col in df.columns:
            df[col] = df[col].map({'t':1,'f':0,True:1,False:0}).fillna(0).astype(int)
    for col in ['minimum_nights','maximum_nights','number_of_reviews',
                'number_of_reviews_ltm','calculated_host_listings_count',
                'availability_365','accommodates']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
    dfs[name] = df[df['price'] > 0].reset_index(drop=True)

print('Types (Lyon) :')
print(dfs['Lyon'].dtypes)
for name, df in dfs.items():
    print(f'{name:10s} : {df.shape}')

In [ ]:
for name, key in CITIES.items():
    dfs[name].to_csv(f'data/processed/{key}_clean.csv', index=False)
print('Donnees nettoyees sauvegardees.')

## 4. Analyse Descriptive des Donnees

In [ ]:
for name, df in dfs.items():
    print(f'\n=== {name} ===')
    display(df[['price','accommodates','bedrooms','beds','bathrooms',
                'review_scores_rating','availability_365']].describe().round(2))

In [ ]:
summary = {}
for name, df in dfs.items():
    summary[name] = {
        'Nb annonces'         : len(df),
        'Prix median (euros)' : round(df['price'].median(), 1),
        'Prix moyen (euros)'  : round(df['price'].mean(), 1),
        'Capacite med. (p.)'  : df['accommodates'].median(),
        'Note moyenne'        : round(df['review_scores_rating'].replace(0,float('nan')).mean(),2),
        'Superhosts (%)'      : round(df['host_is_superhost'].mean()*100, 1),
        'Dispo moy. (j/an)'   : round(df['availability_365'].mean(), 0),
    }
display(pd.DataFrame(summary))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, df) in zip(axes, dfs.items()):
    cap  = df['price'].quantile(0.99)
    data = df[df['price'] <= cap]['price']
    ax.hist(data, bins=60, color=COLORS[name], edgecolor='white', alpha=0.85)
    ax.axvline(data.median(), color='black', linestyle='--',
               label=f'Mediane : {data.median():.0f} euros')
    ax.set_title(f'Distribution des prix - {name}')
    ax.set_xlabel('Prix / nuit (euros)'); ax.legend()
plt.tight_layout()
plt.savefig('reports/figures/prix_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, df) in zip(axes, dfs.items()):
    med = df.groupby('room_type')['price'].median().sort_values()
    ax.barh(med.index, med.values, color=COLORS[name], alpha=0.85)
    ax.set_title(f'Prix median par type - {name}')
    ax.set_xlabel('Prix median (euros)')
    for i, v in enumerate(med.values):
        ax.text(v+1, i, f'{v:.0f}€', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('reports/figures/prix_par_type.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, (name, df) in zip(axes, dfs.items()):
    top = df.groupby('neighbourhood_cleansed')['price'].median().nlargest(10)
    ax.barh(top.index, top.values, color=COLORS[name], alpha=0.85)
    ax.invert_yaxis()
    ax.set_title(f'Top 10 quartiers - {name}')
    ax.set_xlabel('Prix median (euros)')
plt.tight_layout()
plt.savefig('reports/figures/top_quartiers.png', bbox_inches='tight')
plt.show()

## 5. Recodage des Colonnes

Les modeles ML ne comprennent que des **nombres**. On transforme les variables categorielles.


In [ ]:
ROOM_MAP = {'Entire home/apt': 3, 'Hotel room': 2, 'Private room': 1, 'Shared room': 0}

for name, df in dfs.items():
    df['room_type_code'] = df['room_type'].map(ROOM_MAP).fillna(0).astype(int)

print('Encodage room_type :', ROOM_MAP)
display(dfs['Lyon'][['room_type','room_type_code']].drop_duplicates()
        .sort_values('room_type_code', ascending=False))

In [ ]:
for name, df in dfs.items():
    freq = df['neighbourhood_cleansed'].value_counts(normalize=True)
    df['neighbourhood_freq'] = df['neighbourhood_cleansed'].map(freq)

print('Top 5 quartiers Lyon par frequence :')
display(dfs['Lyon'][['neighbourhood_cleansed','neighbourhood_freq']]
    .drop_duplicates().nlargest(5,'neighbourhood_freq'))

In [ ]:
for name, df in dfs.items():
    freq = df['property_type'].value_counts(normalize=True)
    df['property_type_freq'] = df['property_type'].map(freq)

print('Types de proprietes les plus frequents (Lyon) :')
display(dfs['Lyon']['property_type'].value_counts().head(8).to_frame())

## 6. Gestion des Valeurs Aberrantes

Methode des **percentiles 1% - 99%** : on supprime les prix extremes qui fausseraient la droite de regression.


In [ ]:
def remove_outliers(df, col='price', low=0.01, high=0.99):
    q_low, q_high = df[col].quantile(low), df[col].quantile(high)
    n = len(df)
    df = df[(df[col] >= q_low) & (df[col] <= q_high)].copy()
    print(f'  Seuils [{q_low:.0f}€ - {q_high:.0f}€] | Supprimes : {n - len(df)} lignes')
    return df.reset_index(drop=True)

for name in list(dfs.keys()):
    print(f'{name} :')
    dfs[name] = remove_outliers(dfs[name])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, (name, df) in zip(axes, dfs.items()):
    ax.boxplot(df['price'], vert=False, patch_artist=True,
               boxprops=dict(facecolor=COLORS[name], alpha=0.7))
    ax.set_title(f'Boxplot prix - {name}')
    ax.set_xlabel('Prix (euros)')
plt.tight_layout()
plt.savefig('reports/figures/boxplot_prix.png', bbox_inches='tight')
plt.show()

In [ ]:
for name, key in CITIES.items():
    dfs[name].to_csv(f'data/processed/{key}_features.csv', index=False)
print('Donnees finales sauvegardees :')
for name, df in dfs.items():
    print(f'  {name:10s} : {df.shape}')

## 7. Separation Train/Test & Modelisation

**Separation 80% / 20%** : le modele apprend sur 80% des donnees, on evalue sur les 20% restants (jamais vus pendant l'entrainement).


In [ ]:
SIMPLE_FEAT = ['accommodates']

MULTIPLE_FEAT = [
    'accommodates', 'bedrooms', 'beds', 'bathrooms',
    'minimum_nights', 'availability_365',
    'number_of_reviews', 'reviews_per_month',
    'review_scores_rating', 'review_scores_cleanliness', 'review_scores_location',
    'host_is_superhost', 'host_response_rate', 'calculated_host_listings_count',
    'instant_bookable', 'room_type_code', 'neighbourhood_freq', 'property_type_freq',
]

def fit_eval(df, features, label=''):
    feats = [f for f in features if f in df.columns]
    d = df[feats + [TARGET]].dropna()
    X, y = d[feats], d[TARGET]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LinearRegression().fit(Xtr, ytr)
    yp    = model.predict(Xte)
    mae   = round(mean_absolute_error(yte, yp), 2)
    rmse  = round(np.sqrt(mean_squared_error(yte, yp)), 2)
    r2    = round(r2_score(yte, yp), 4)
    return model, Xte, yte, yp, feats, mae, rmse, r2

results = {}
print('=== Entrainement des modeles ===')
for name, df in dfs.items():
    print(f'\n{name}  (train={int(len(df)*0.8):,}  test={int(len(df)*0.2):,})')
    results[f'{name}_simple']   = fit_eval(df, SIMPLE_FEAT,   'Simple')
    results[f'{name}_multiple'] = fit_eval(df, MULTIPLE_FEAT, 'Multiple')
    for label in ['simple','multiple']:
        r = results[f'{name}_{label}']
        print(f'  {label:8s} -> MAE={r[5]}€  RMSE={r[6]}€  R2={r[7]}')

### 7a. Regression Lineaire Simple

Variable unique : `accommodates` (nombre de personnes).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, name in zip(axes, CITIES):
    m, Xte, yte, yp, feats, mae, rmse, r2 = results[f'{name}_simple']
    ax.scatter(Xte['accommodates'], yte, alpha=0.25, s=8, color=COLORS[name], label='Reel')
    xs = sorted(Xte['accommodates'].unique())
    ax.plot(xs, m.predict(pd.DataFrame({'accommodates': xs})),
            color='black', linewidth=2, label='Droite')
    ax.set_title(f'Regression simple - {name}\nMAE={mae}€  R2={r2}')
    ax.set_xlabel('Nb personnes (accommodates)')
    ax.set_ylabel('Prix (euros)'); ax.legend()
plt.tight_layout()
plt.savefig('reports/figures/regression_simple.png', bbox_inches='tight')
plt.show()

### 7b. Regression Lineaire Multiple

18 variables : capacite, chambres, scores, hote, disponibilite...

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, name in zip(axes, CITIES):
    m, Xte, yte, yp, feats, mae, rmse, r2 = results[f'{name}_multiple']
    ax.scatter(yte, yp, alpha=0.2, s=8, color=COLORS[name])
    lims = [min(yte.min(), yp.min()), max(yte.max(), yp.max())]
    ax.plot(lims, lims, 'k--', linewidth=1, label='Prediction parfaite')
    ax.set_title(f'Reel vs Predit - {name}\nMAE={mae}€  R2={r2}')
    ax.set_xlabel('Prix reel (euros)'); ax.set_ylabel('Prix predit (euros)')
    ax.legend()
plt.tight_layout()
plt.savefig('reports/figures/regression_multiple.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, name in zip(axes, CITIES):
    m, _, _, _, feats, _, _, _ = results[f'{name}_multiple']
    coefs = pd.Series(m.coef_, index=feats).sort_values()
    ax.barh(coefs.index, coefs.values,
            color=['coral' if v < 0 else COLORS[name] for v in coefs])
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Coefficients - {name}')
    ax.set_xlabel('Coefficient')
plt.tight_layout()
plt.savefig('reports/figures/coefficients.png', bbox_inches='tight')
plt.show()

## 8. Evaluation & Prediction

In [ ]:
rows = []
for name in CITIES:
    for label in ['simple', 'multiple']:
        _, _, yte, yp, _, mae, rmse, r2 = results[f'{name}_{label}']
        rows.append({
            'Ville': name, 'Modele': label.capitalize(),
            'MAE (euros)': mae, 'RMSE (euros)': rmse, 'R2': r2,
            'Nb test': len(yte),
        })
display(pd.DataFrame(rows))

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 14))
for row, name in enumerate(CITIES):
    for col, label in enumerate(['simple','multiple']):
        _, _, yte, yp, _, mae, _, r2 = results[f'{name}_{label}']
        res = yte.values - yp
        ax  = axes[row][col]
        ax.scatter(yp, res, alpha=0.25, s=8, color=COLORS[name])
        ax.axhline(0, color='black', linewidth=1, linestyle='--')
        ax.set_title(f'Residus - {name} ({label})  MAE={mae}€')
        ax.set_xlabel('Prix predit (euros)'); ax.set_ylabel('Residu (euros)')
plt.tight_layout()
plt.savefig('reports/figures/residus.png', bbox_inches='tight')
plt.show()

### Prediction pour un nouveau logement

Simulation : **T2, 4 personnes, note 4.8, superhost, appartement entier**.


In [ ]:
new_listing = dict(
    accommodates=4, bedrooms=2, beds=2, bathrooms=1.0,
    minimum_nights=2, availability_365=200,
    number_of_reviews=25, reviews_per_month=1.5,
    review_scores_rating=4.8, review_scores_cleanliness=4.9,
    review_scores_location=4.7, host_is_superhost=1,
    host_response_rate=95.0, calculated_host_listings_count=1,
    instant_bookable=1, room_type_code=3,
    neighbourhood_freq=0.05, property_type_freq=0.4,
)

print('Logement : T2, 4 personnes, note 4.8, superhost, appartement entier\n')
for name in CITIES:
    m, _, _, _, feats, _, _, _ = results[f'{name}_multiple']
    X_new = pd.DataFrame([[new_listing.get(f, 0) for f in feats]], columns=feats)
    prix  = round(float(m.predict(X_new)[0]), 2)
    print(f'  {name:10s} -> {prix} euros/nuit')

## Conclusion

- Le **nombre de personnes** (`accommodates`) est la variable la plus correlee au prix
- Les **scores de review** et le statut **superhost** ameliorent nettement les predictions
- La **regression multiple** surpasse la simple sur les 3 villes
- **Bordeaux** presente le meilleur R2 (0.62) — marche plus homogene
- **Paris** a les prix les plus eleves et la variance la plus grande

| Ville | MAE Simple | R2 Simple | MAE Multiple | R2 Multiple |
|-------|-----------|-----------|-------------|-------------|
| Lyon | 36.46€ | 0.27 | 32.94€ | **0.38** |
| Paris | 117.52€ | 0.23 | 105.31€ | **0.33** |
| Bordeaux | 38.08€ | 0.51 | 32.54€ | **0.62** |

**Pistes d'amelioration** : Random Forest, XGBoost, donnees geographiques (distance centre-ville), saisonnalite.
